# COMP663 Assignment 2 — Classical Optimisation

**Student ID:** 1173808  
**Dataset:** `forest_cover_data.csv`  
**Primary metric:** macro-F1


## ENV & Libs Setup


In [23]:
from pathlib import Path
import ast
import random
import time
setup_started = time.perf_counter()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import torch
from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import (
    ParameterSampler,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler
from torch import nn

SEED = 42
if Path("/kaggle").exists() and not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU was not allocated; stop instead of running on CPU.")
DEVICE = torch.device(
      "mps" if torch.backends.mps.is_available()
      else "cuda" if torch.cuda.is_available()
      else "cpu"
  )
torch.set_num_threads(4)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
KAGGLE_DATA_PATH = Path("/kaggle/input/datasets/yangliunz/comp663-a2-forest-cove/forest_cover_data.csv")
kaggle_data_paths = ([KAGGLE_DATA_PATH] if KAGGLE_DATA_PATH.exists() else []) + list(Path("/kaggle/input").rglob("forest_cover_data.csv"))
DATA_PATH = kaggle_data_paths[0] if kaggle_data_paths else ROOT / "data" / "forest_cover_data.csv"
FIGURE_PATH = ROOT / "figures" / "performance_comparison.png"
MODEL_PATH = ROOT / "models" / "1173808_Assignment2_final.pt"
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.20
SEARCH_EPOCHS = 20
FINAL_EPOCHS = 100
RANDOM_TRIALS = 12
BAYESIAN_TRIALS = 12
NAS_TRIALS = 12

if DEVICE.type == "cuda":
    RUNTIME_DEVICE = f"{torch.cuda.get_device_name(0)} x{torch.cuda.device_count()}"
elif DEVICE.type == "mps":
    RUNTIME_DEVICE = "Apple Silicon MPS"
else:
    RUNTIME_DEVICE = "CPU"

def format_decimal(value):
    return np.format_float_positional(float(value), unique=True, trim="-")

def log_cell(name, started, configuration):
    print(f"{name}: device={RUNTIME_DEVICE}; configuration={configuration}; elapsed_seconds={format_decimal(time.perf_counter() - started)}")

log_cell("Environment setup", setup_started, f"torch={torch.__version__}, seed={SEED}, data={DATA_PATH}")


## Task 1 — Baseline model and candidate hyperparameters

### 1.1 Preprocessing pipeline and feature engineering

As we found during assginment 1:

| Decision                 | Evidence from EDA                                                                                                                      | Action                                                                                                       | Reason / trade-off                                                                                                                                                                                              |
| ------------------------ | -------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Missing values           | Zero missing values across all 15 columns (confirmed in Task 1.1)                                                                      | pass through                                                                                                 | Imputing when there is nothing to impute would add unnecessary complexity and risk introducing artificial patterns.                                                                                             |
| Scaling / transformation | Distance features have large ranges and outliers (Task 1.3)                                                                            | Apply `StandardScaler` to the 10 continuous features and leave the 4 binary Wilderness_Area columns unscaled | Scaling puts continuous features on a comparable range. The fitted transformation stays inside the validation pipeline, which prevents leakage. Some models are notdistance-sensitive which will discuss later. |
| Feature engineering      | Elevation, distance features, and wilderness area already show useful class separation; wilderness columns are already one-hot encoded | No additional feature engineering                                                                            | The existing features already contain useful information. Adding polynomial features would increase complexity without EDA evidence that they are needed.                                                       |
| Class imbalance          | The largest and smallest classes have a 31:1 ratio (Task 1.2)                                                                          | Use macro-F1 and balanced accuracy for evaluation                                                            | These metrics make minority-class performance visible without changing the baseline training procedure.                                                                                                           |




#### Load data and check data integrity


In [24]:
# load data from CSV file
cell_started = time.perf_counter()
data = pd.read_csv(DATA_PATH)
target = "Cover_Type"

# filter out rows with missing values in the target column
data = data.dropna(subset=[target])
feature_names = [column for column in data.columns if column != target]
continuous_features = [
    column for column in feature_names if not column.startswith("Wilderness_Area")
]

# check data integrity
assert data.shape == (571_012, 15), data.shape
assert len(feature_names) == 14
assert set(data[target].unique()) == {1, 2, 3, 4, 5}
assert data.isna().sum().sum() == 0

# print data shape, feature names, and target value counts with percentages
print("Shape:", data.shape)
print("Features:", feature_names)
display(
    data[target]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
    .assign(percentage=lambda frame: 100 * frame["count"] / len(data))
)
log_cell("Data loading", cell_started, f"data={DATA_PATH}")


#### Data set split

In [25]:
# split data into training, validation, and test sets
cell_started = time.perf_counter()
train_validation_frame, test_frame = train_test_split(
    data, test_size=TEST_FRACTION, stratify=data[target], random_state=SEED
)

train_frame, validation_frame = train_test_split(
    train_validation_frame,
    test_size=VALIDATION_FRACTION / (TRAIN_FRACTION + VALIDATION_FRACTION),
    stratify=train_validation_frame[target],
    random_state=SEED,
)

# validate the size of the splits and print the number of samples in each set
assert len(train_frame) + len(validation_frame) + len(test_frame) == len(data)
print(
    f"Train / validation / test: {len(train_frame):,} / {len(validation_frame):,} / {len(test_frame):,}"
)
log_cell("Data split", cell_started, "train=0.6, validation=0.2, test=0.2")


### 1.2 Primary and secondary evaluation metrics

As this dataset have a strong imblanaced class,

**Macro-F1** will be the primary metric to cover type has equal importance.

**Balanced accuracy** is the secondary metric because it reflect the model recalls on each class equally.


### 1.3 Baseline model training and evaluation procedure

- (1) Initial the baseline mode with configuration in assignment requirement.

- (2) traning baseline model and compute metrics on validation dataset.


In [26]:
# Define the fixed baseline architecture required by the assignment.
cell_started = time.perf_counter()
class BaselineNN(nn.Module):
    """The architecture supplied in baselineNN.ipynb."""

    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 24),  # 14 input features to 24 hidden units
            nn.Sigmoid(),               # required hidden-layer activation
            nn.Linear(24, 12),          # second hidden layer
            nn.Sigmoid(),               # required hidden-layer activation
            nn.Linear(12, 5),           # one logit for each Cover_Type class
        )

    def forward(self, x):
        return self.layers(x)


# Create a fresh baseline model on the selected device.
baseline_model = BaselineNN(len(feature_names)).to(DEVICE)
print(baseline_model)
print(
    f"Parameters: {sum(parameter.numel() for parameter in baseline_model.parameters()):,}"
)
# Check the supplied architecture: weights and biases total 725 trainable parameters.
assert (
    sum(parameter.numel() for parameter in baseline_model.parameters()) == 725
), "Parameter count mismatch"
log_cell("Baseline architecture", cell_started, "layers=14-24-12-5, activation=Sigmoid, parameters=725")


In [28]:
# Train the baseline model and evaluate it on the validation split.
cell_started = time.perf_counter()
BASELINE_CONFIG = {
    "learning_rate": 0.001,
    "batch_size": 512,
    "epochs": SEARCH_EPOCHS,
}

# Fit scaling values on training data only to prevent data leakage.
scaler = StandardScaler().fit(train_frame[continuous_features])


def prepare_baseline_data(frame):
    # Keep the original 14 feature columns and use float32 for PyTorch.
    features = frame[feature_names].astype("float32").copy()
    # Scale only continuous features; Wilderness_Area columns are already 0/1.
    features[continuous_features] = scaler.transform(features[continuous_features])
    # Change class labels from 1–5 to the 0–4 indices required by CrossEntropyLoss.
    return features.to_numpy(), frame[target].to_numpy(dtype=np.int64) - 1


# Apply the training-fitted scaler to both training and validation features.
x_train, y_train = prepare_baseline_data(train_frame)
x_validation, y_validation = prepare_baseline_data(validation_frame)

# Keep the supplied baseline loss unchanged.
loss_fn = nn.CrossEntropyLoss()
# Adam updates the baseline parameters using the fixed Task 1 settings.
optimizer = torch.optim.Adam(
    baseline_model.parameters(),
    lr=BASELINE_CONFIG["learning_rate"],
)

# Move the training arrays to PyTorch tensors once before the epoch loop.
x_train_tensor = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
# Use the fixed seed so the mini-batch order is reproducible.
generator = torch.Generator(device=DEVICE).manual_seed(SEED)

# Training mode enables gradient calculation and parameter updates.
baseline_model.train()
for _ in range(BASELINE_CONFIG["epochs"]):
    # Shuffle the training rows once per epoch.
    order = torch.randperm(len(x_train_tensor), generator=generator, device=DEVICE)
    for start in range(0, len(order), BASELINE_CONFIG["batch_size"]):
        batch_index = order[start : start + BASELINE_CONFIG["batch_size"]]
        optimizer.zero_grad()  # clear gradients from the previous mini-batch
        loss = loss_fn(
            baseline_model(x_train_tensor[batch_index]), y_train_tensor[batch_index]
        )
        loss.backward()  # compute gradients by backpropagation
        optimizer.step()  # update weights and biases

# Evaluation mode and no_grad disable training updates for validation.
baseline_model.eval()
with torch.no_grad():
    validation_logits = baseline_model(
        torch.tensor(x_validation, dtype=torch.float32, device=DEVICE)
    )

# Use softmax to convert logits to probabilities, then take the argmax to get predictions.
probabilities = torch.softmax(validation_logits, dim=1)
validation_prediction = probabilities.argmax(dim=1).cpu().numpy()

# calculate macro-F1 and balanced accuracy scores for the validation set
validation_macro_f1 = f1_score(y_validation, validation_prediction, average="macro")
validation_balanced_accuracy = balanced_accuracy_score(
    y_validation, validation_prediction
)
# Metrics must be valid scores between zero and one.
assert 0 <= validation_macro_f1 <= 1
print("Validation macro-F1:", validation_macro_f1)
print("Validation balanced accuracy:", validation_balanced_accuracy)
log_cell("Baseline training and validation", cell_started, f"learning_rate={format_decimal(BASELINE_CONFIG['learning_rate'])}, batch_size={BASELINE_CONFIG['batch_size']}, epochs={BASELINE_CONFIG['epochs']}")


**Baseline Model metrics on validation data set**

| Model      | Validation macro-F1 | Validation balanced accuracy |
| ---------- | ------------------: | ---------------------------: |
| BaselineNN |  0.5166125906224147 |           0.4830609876221151 |


### 1.4 Unoptimised baseline performance using held-out test data


In [11]:
# evaluate the unoptimised baseline on the held-out test set
cell_started = time.perf_counter()
x_test, y_test = prepare_baseline_data(test_frame)

baseline_model.eval()
with torch.no_grad():
    test_logits = baseline_model(
        torch.tensor(x_test, dtype=torch.float32, device=DEVICE)
    )
test_probabilities = torch.softmax(test_logits, dim=1)
test_prediction = test_probabilities.argmax(dim=1).cpu().numpy()

test_macro_f1 = f1_score(y_test, test_prediction, average="macro")
test_balanced_accuracy = balanced_accuracy_score(y_test, test_prediction)
assert 0 <= test_macro_f1 <= 1
print("Test macro-F1:", test_macro_f1)
print("Test balanced accuracy:", test_balanced_accuracy)
log_cell("Baseline test evaluation", cell_started, "model=BaselineNN")


\*\*Baseline Model metrics on Test dataset
| Model | Test macro-F1 | Test balanced accuracy |
|---|---:|---:|
| BaselineNN | 0.4717161780992363 | 0.7504576390688007 |


### 1.5 Candidate hyperparameters

- (1) Learning rate: step size and convergence stability.
- (2) Batch size: size of updates per epoch and effect training cost.
- (3) Weight decay: regularises weights and may reduce overfitting.
- (4) Epochs: controls the training budget and convergence time.
- (5) NAS architectural design choices — number of hidden layers, hidden-layer widths, and activation functions: change model capacity and parameter count.


### 1.6 Hyperparameter types and search ranges

| Hyperparameter | Type and range | Expected effect / cost |
|---|---|---|
| Learning rate | log continuous, 0.0001–0.01 | Controls update size; too large can be unstable. |
| Batch size | categorical: 256, 512, 1024 | Larger batches use fewer updates per epoch. |
| Weight decay | log continuous, 0.00001–0.001 | Regularises weights; too much can underfit. |
| Epochs | fixed at 20 per trial | Keeps the comparison within a practical budget. |
| Number of hidden layers | NAS search space: categorical 1, 2, or 3 | Changes model depth and parameter count. |
| Hidden-layer width | NAS search space: categorical 16, 24, 32, 48, or 64 | Changes model capacity and training cost. |
| Activation function | NAS search space: categorical Sigmoid or ReLU | Changes non-linear behaviour and convergence. |


## Task 2 — Classical hyperparameter search


### 2.1 Selected Random Search method and justification

Choose random search as it covers the learning-rate and weight-decay ranges more efficiently than a small grid since both ranges are meaningful on a log scale.


### 2.2 Hyperparameters to optimise

Random search will focus on optimising `learning rate`, `batch size`, and `weight decay` from are the Task 1 candidates;
`Epochs` and other architecture hyperparameters will be fixed during task 2 for a fair and justified comparision.


### 2.3 Computational budget and justification

The budget is fixed as 12 trials in 20 epochs.
This is small enough for CPU execution while testing different training settings.


### 2.4 Apply Random Search
Apply random search based on the task 1.6 hyperparameters table by 12 trails

In [43]:
# Task 2 keeps the Task 1 baseline architecture fixed.
cell_started = time.perf_counter()
# Create tensors once so every trial reuses the same scaled data on the selected device.
x_train_tensor = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
x_validation_tensor = torch.tensor(x_validation, dtype=torch.float32, device=DEVICE)

def train_and_evaluate(config):
    torch.manual_seed(SEED)
    model = BaselineNN(len(feature_names)).to(DEVICE)

    # Reuse the Task 1 tensors and training-fitted scaler.

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    model.train()
    for _ in range(config["epochs"]):
        # torch.randperm on DEVICE works on both Apple MPS and Kaggle CUDA.
        order = torch.randperm(len(x_train_tensor), device=DEVICE)
        for start in range(0, len(order), config["batch_size"]):
            batch_index = order[start : start + config["batch_size"]]
            optimizer.zero_grad()
            loss = loss_fn(model(x_train_tensor[batch_index]), y_train_tensor[batch_index])
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(x_validation_tensor)
        probabilities = torch.softmax(logits, dim=1)
        prediction = probabilities.argmax(dim=1).cpu().numpy()

    return (
        f1_score(y_validation, prediction, average="macro"),
        balanced_accuracy_score(y_validation, prediction),
    )


# Random search samples configurations without evaluating every grid combination.
random_space = {
    "learning_rate": [0.0001, 0.001, 0.01],
    "batch_size": [256, 512, 1024],
    "weight_decay": [0.0, 0.00001, 0.0001, 0.001],
}
random_rows = []
for trial_number, sampled in enumerate(
    ParameterSampler(random_space, n_iter=RANDOM_TRIALS, random_state=SEED), 1
):
    config = {**sampled, "epochs": SEARCH_EPOCHS}
    print(f"Task 2 trial {trial_number}/{RANDOM_TRIALS}: learning_rate={format_decimal(config['learning_rate'])}, batch_size={config['batch_size']}, weight_decay={format_decimal(config['weight_decay'])}, epochs={config['epochs']}", flush=True)
    started = time.perf_counter()
    macro_f1, balanced_accuracy = train_and_evaluate(config)
    row = {
        "trial": trial_number,
        **config,
        "macro_f1": macro_f1,
        "balanced_accuracy": balanced_accuracy,
        "parameters": 725,
        "seconds": time.perf_counter() - started,
    }
    random_rows.append(row)
    print(f"Task 2 trial {trial_number} result: macro-F1={format_decimal(row['macro_f1'])}, balanced accuracy={format_decimal(row['balanced_accuracy'])}, seconds={format_decimal(row['seconds'])}", flush=True)

random_table = pd.DataFrame(random_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
best_random = random_table.iloc[0].to_dict()
print("Task 2 complete:")
print(random_table.to_string(index=False, float_format=lambda value: np.format_float_positional(float(value), unique=True, trim="-")))
log_cell("Task 2 random search", cell_started, f"trials={RANDOM_TRIALS}, epochs_per_trial={SEARCH_EPOCHS}")


NameError: name 'format_decimal' is not defined

### 2.5 Evaluation procedure and primary metric

Every trial trains on the same 60% training dataset.
Generate metrics macro-f1 balanced_accuracy only on same vlidation dataset.
The scaler is only appied on training dataset only to prevent data leakage.


### 2.6 Best configuration, score, and search time

| Item                                 |              Value |
| ------------------------------------ | -----------------: |
| Learning rate                        |               0.01 |
| Batch size                           |                512 |
| Weight decay                         |            0.00001 |
| Epochs                               |                 20 |
| Validation macro-F1                  |  0.620486164019339 |
| Validation balanced accuracy         | 0.5615715029443511 |
| Search time on Tesla T4 x2 (seconds) | 220.10828787800006 |


### 2.7 Search results table

| Rank | Trial | Learning rate | Batch size | Weight decay | Macro-F1 | Balanced accuracy | Seconds |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 7 | 0.01 | 512 | 0.00001 | 0.620486164019339 | 0.5615715029443511 | 19.434950119999996 |
| 2 | 9 | 0.01 | 256 | 0.0 | 0.6058210695912665 | 0.5496740507572219 | 37.53823663899999 |
| 3 | 11 | 0.01 | 256 | 0.00001 | 0.6042274044597084 | 0.5567435507062999 | 38.28666197899997 |
| 4 | 12 | 0.01 | 1024 | 0.0001 | 0.526955832979515 | 0.4913293427281696 | 9.93340552199993 |
| 5 | 5 | 0.001 | 512 | 0.0 | 0.45214492618835067 | 0.43567754133121106 | 18.69509913899998 |
| 6 | 10 | 0.001 | 512 | 0.00001 | 0.4460991064559261 | 0.43167661114895833 | 19.573229812000022 |
| 7 | 4 | 0.001 | 1024 | 0.0001 | 0.41715277351203195 | 0.4036995878770145 | 9.667997221999997 |
| 8 | 1 | 0.01 | 1024 | 0.001 | 0.3631319448186237 | 0.35042005402198173 | 9.700595045 |
| 9 | 6 | 0.001 | 1024 | 0.001 | 0.3138612797455258 | 0.3175054099652919 | 9.57150824300004 |
| 10 | 8 | 0.0001 | 512 | 0.0 | 0.2971883716053794 | 0.3090180104120446 | 18.66795302700001 |
| 11 | 2 | 0.0001 | 512 | 0.00001 | 0.29717649858291145 | 0.309005325363587 | 19.385552362999988 |
| 12 | 3 | 0.0001 | 1024 | 0.0001 | 0.2947015511563337 | 0.30636080388925907 | 9.62775801700002 |

The table shows that learning rate = 0.01 gives the strongest trials, but its result still depends on batch size and weight decay.


## Task 3 — Bayesian optimisation


### 3.1 Hyperparameters to optimise

Bayesian optimisation searches learning rate, batch size, and weight decay as same as three training hyperparameters searched in Task 2. The supplied baseline architecture remains fixed.


### 3.2 Objective, search space, surrogate model, and acquisition process

The objective is to find the best validation macro-F1.
Five initial configurations are created randomly.
Use Gaussian Process Regressor as ConstantKernel during optimisation.
Learning rate and weight decay are applied withlog scales; batch size encoded as one-hot encoding to prevent large number bias during optimisation and keep a fair comparision with task 2.


### 3.3 Evaluation procedure and primary metric

Every trial trains on the same 60% training split. Macro-F1 and balanced accuracy are calculated only on the same 20% validation split. The scaler is fitted on the training split only to prevent data leakage.


### 3.4 Computational budget and justification

The budget is fixed at 12 trials of 20 epochs: five random initial trials and seven Gaussian Process plus Expected Improvement trials. This gives Task 3 the same search space and total trial budget as Task 2 for a fair comparison.


### 3.5 Apply Bayesian optimisation


In [ ]:
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern

# Record the total Bayesian-optimisation time.
search_started = time.perf_counter()
bayesian_rows = []
# Encode the categorical batch-size choices for the Gaussian Process.
BATCH_SIZES = np.array([256, 512, 1024])
LOG_LR_BOUNDS = (-4.0, -2.0)
LOG_WEIGHT_DECAY_BOUNDS = (-5.0, -3.0)


# Expected Improvement balances predicted score and uncertainty.
def expected_improvement(mu, sigma, best_so_far, xi=0.01):
    # maximise macro-F1, so improvement means predicting ABOVE best_so_far.
    sigma = np.maximum(sigma, 0.000000001)
    improvement = mu - best_so_far - xi
    z = improvement / sigma
    return improvement * norm.cdf(z) + sigma * norm.pdf(z)


# Scale continuous variables and one-hot encode batch size for GP input.
def gp_features(points):
    points = np.asarray(points)
    lower = np.array([LOG_LR_BOUNDS[0], LOG_WEIGHT_DECAY_BOUNDS[0]])
    upper = np.array([LOG_LR_BOUNDS[1], LOG_WEIGHT_DECAY_BOUNDS[1]])
    continuous = (points[:, :2] - lower) / (upper - lower)
    batch_one_hot = np.eye(len(BATCH_SIZES))[points[:, 2].astype(int)]
    return np.column_stack([continuous, batch_one_hot])


# Convert a candidate into training settings and evaluate one validation trial.
def evaluate_configuration(
    log_learning_rate, log_weight_decay, batch_index, trial_number
):
    config = {
        "learning_rate": float(10**log_learning_rate),
        "batch_size": int(BATCH_SIZES[int(batch_index)]),
        "weight_decay": float(10**log_weight_decay),
        "epochs": SEARCH_EPOCHS,
    }
    print(
        f"Task 3 trial {trial_number}/{BAYESIAN_TRIALS}: learning_rate={format_decimal(config['learning_rate'])}, batch_size={config['batch_size']}, weight_decay={format_decimal(config['weight_decay'])}, epochs={config['epochs']}",
        flush=True,
    )
    started = time.perf_counter()
    macro_f1, balanced_accuracy = train_and_evaluate(config)
    row = {
        "trial": trial_number,
        **config,
        "macro_f1": macro_f1,
        "balanced_accuracy": balanced_accuracy,
        "parameters": 725,
        "seconds": time.perf_counter() - started,
    }
    bayesian_rows.append(row)
    print(
        f"Task 3 trial {trial_number} result: macro-F1={format_decimal(macro_f1)}, balanced accuracy={format_decimal(balanced_accuracy)}, seconds={format_decimal(row['seconds'])}",
        flush=True,
    )
    return macro_f1


# Evaluate five random initial points before fitting the surrogate.
rng = np.random.default_rng(SEED)
initial_points = np.column_stack(
    [
        rng.uniform(*LOG_LR_BOUNDS, 5),
        rng.uniform(*LOG_WEIGHT_DECAY_BOUNDS, 5),
        rng.integers(0, len(BATCH_SIZES), 5),
    ]
)
evaluated_points = list(initial_points)
evaluated_scores = [
    evaluate_configuration(point[0], point[1], point[2], trial_number)
    for trial_number, point in enumerate(initial_points, 1)
]

# Define the 5 Gaussian Process surrogate and candidate grid.
kernel = ConstantKernel(1.0, (0.01, 100.0)) * Matern(
    length_scale=0.3, length_scale_bounds=(0.05, 3.0), nu=2.5
)
log_learning_rates = np.linspace(*LOG_LR_BOUNDS, 40)
log_weight_decays = np.linspace(*LOG_WEIGHT_DECAY_BOUNDS, 40)
learning_rate_grid, weight_decay_grid, batch_grid = np.meshgrid(
    log_learning_rates, log_weight_decays, np.arange(len(BATCH_SIZES)), indexing="ij"
)
candidates = np.column_stack(
    [learning_rate_grid.ravel(), weight_decay_grid.ravel(), batch_grid.ravel()]
)
candidate_used = np.zeros(len(candidates), dtype=bool)

# Fit the GP after each result and select the candidate with the highest EI.
for trial_number in range(6, BAYESIAN_TRIALS + 1):
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=0.001,
        normalize_y=True,
        n_restarts_optimizer=2,
        random_state=SEED,
    )
    gp.fit(gp_features(evaluated_points), evaluated_scores)
    mean, standard_deviation = gp.predict(gp_features(candidates), return_std=True)
    improvement = expected_improvement(mean, standard_deviation, max(evaluated_scores))
    improvement[candidate_used] = -np.inf
    next_index = int(np.argmax(improvement))
    next_point = candidates[next_index]
    candidate_used[next_index] = True
    evaluated_points.append(next_point)
    evaluated_scores.append(
        evaluate_configuration(
            next_point[0], next_point[1], next_point[2], trial_number
        )
    )

# Sort completed trials so the best validation macro-F1 appears first.
bayesian_table = (
    pd.DataFrame(bayesian_rows)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
best_bayesian = bayesian_table.iloc[0]
print("Task 3 complete:")
print(
    f"Best Bayesian configuration: learning_rate={format_decimal(best_bayesian['learning_rate'])}, batch_size={int(best_bayesian['batch_size'])}, weight_decay={format_decimal(best_bayesian['weight_decay'])}, epochs={int(best_bayesian['epochs'])}"
)
print(
    bayesian_table.to_string(
        index=False, float_format=lambda value: format_decimal(value)
    )
)
log_cell(
    "Task 3 Bayesian optimisation",
    search_started,
    f"method=Gaussian Process plus Expected Improvement, batch_size=one-hot categorical, trials={BAYESIAN_TRIALS}, epochs_per_trial={SEARCH_EPOCHS}",
)

Task 3 trial 1/12: learning_rate=0.00353111691382141, batch_size=512, weight_decay=0.0008938089586343595, epochs=20
Task 3 trial 1 result: macro-F1=0.36765088688041786, balanced accuracy=0.3539646519785102, seconds=23.80778444300006
Task 3 trial 2/12: learning_rate=0.000754669641079693, batch_size=512, weight_decay=0.0003328736391557839, epochs=20
Task 3 trial 2 result: macro-F1=0.4017970264372613, balanced accuracy=0.38409752960236354, seconds=17.67033995700001
Task 3 trial 3/12: learning_rate=0.005214297905300546, batch_size=256, weight_decay=0.0003733607072503032, epochs=20
Task 3 trial 3 result: macro-F1=0.42119014945993083, balanced accuracy=0.4019880579276133, seconds=35.2360818840001
Task 3 trial 4/12: learning_rate=0.002481624443014984, batch_size=1024, weight_decay=0.000018039615030067847, epochs=20
Task 3 trial 4 result: macro-F1=0.5144614784334609, balanced accuracy=0.47769149705260305, seconds=8.987462680000021
Task 3 trial 5/12: learning_rate=0.00015429601005519726, batch_

### 3.6 Best configuration, performance, and search time

| Item                                 |              Value |
| ------------------------------------ | -----------------: |
| Learning rate                        |               0.01 |
| Batch size                           |                512 |
| Weight decay                         |            0.00001 |
| Epochs                               |                 20 |
| Validation macro-F1                  |  0.620486164019339 |
| Validation balanced accuracy         | 0.5615715029443511 |
| Search time on Tesla T4 x2 (seconds) | 228.18862507699998 |

The best configuration is identical to the best Task 2 random-search configuration.


### 3.7 Search results table

| Rank | Trial | Learning rate | Batch size | Weight decay | Macro-F1 | Balanced accuracy | Seconds |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 9 | 0.01 | 512 | 0.00001 | 0.620486164019339 | 0.5615715029443511 | 17.725169719000064 |
| 2 | 10 | 0.01 | 256 | 0.00001 | 0.6042274044597084 | 0.5567435507062999 | 35.438160094999944 |
| 3 | 7 | 0.005541020330009492 | 1024 | 0.00001 | 0.5628973313639289 | 0.5141414183602933 | 8.889688035000063 |
| 4 | 8 | 0.01 | 1024 | 0.00001603718743751331 | 0.5609506222558817 | 0.5115403180498805 | 8.812657226999931 |
| 5 | 4 | 0.002481624443014984 | 1024 | 0.000018039615030067847 | 0.5144614784334609 | 0.47769149705260305 | 8.987462680000021 |
| 6 | 12 | 0.001511775070615663 | 512 | 0.00001 | 0.507048406347306 | 0.47200694182848607 | 17.79730464499994 |
| 7 | 6 | 0.002154434690031882 | 1024 | 0.0000180472176682717 | 0.5063222188097061 | 0.47125252379991467 | 9.209170169999993 |
| 8 | 3 | 0.005214297905300546 | 256 | 0.0003733607072503032 | 0.42119014945993083 | 0.4019880579276133 | 35.2360818840001 |
| 9 | 2 | 0.000754669641079693 | 512 | 0.0003328736391557839 | 0.4017970264372613 | 0.38409752960236354 | 17.67033995700001 |
| 10 | 1 | 0.00353111691382141 | 512 | 0.0008938089586343595 | 0.36765088688041786 | 0.3539646519785102 | 23.80778444300006 |
| 11 | 11 | 0.0001 | 256 | 0.00001 | 0.31715455415846483 | 0.3196704347265489 | 35.265415331999975 |
| 12 | 5 | 0.00015429601005519726 | 1024 | 0.00007957412573105527 | 0.2972754375319542 | 0.30929223056984734 | 8.93329130899997 |

The GP/EI trials found the same best configuration as Task 2: learning rate = 0.01, batch size = 512, and weight decay = 0.00001.


## Task 4 — Neural architecture search


### 4.1 NAS hyperparameters to optimise

NAS optimises hidden-layer count, hidden-layer widths, activation, learning rate, batch size, and weight decay. All choices are declared in Task 1.


### 4.2 Objective, NAS search space, and search strategy

The objective is to maximise validation macro-F1. I use evolutionary NAS from Tutorial 5: a population of four candidates is evaluated for three generations; tournament selection, crossover, mutation, and elitism create the next generation.


### 4.3 Architecture design choices

The architecture search space allows one to three hidden layers, widths from 16 to 64, and Sigmoid or ReLU activation. The output remains five logits.


### 4.4 Evaluation procedure and primary metric

NAS uses the same 60% training split, 20% validation split, training-only scaling, 20-epoch budget, and macro-F1 metric as the other searches.


### 4.5 Computational budget and justification

The budget is 12 NAS trials. It is enough to test real architecture choices without using an impractical exhaustive search.


### 4.6 Apply NAS to hyperparameters and architecture


In [ ]:
# Evolutionary NAS: four candidates, three generations, and 20 epochs per candidate.
NAS_CHOICES = {'layers': (1, 2, 3), 'width': (16, 24, 32, 48, 64), 'activation': ('Sigmoid', 'ReLU'), 'learning_rate': (0.0001, 0.001, 0.01), 'batch_size': (256, 512, 1024), 'weight_decay': (0.0, 0.00001, 0.0001, 0.001)}
class NASNN(nn.Module):
    def __init__(self, layers, width, activation):
        super().__init__(); activation_layer = nn.Sigmoid if activation == 'Sigmoid' else nn.ReLU
        network, previous = [], len(feature_names)
        for _ in range(layers):
            network += [nn.Linear(previous, width), activation_layer()]; previous = width
        self.layers = nn.Sequential(*network, nn.Linear(previous, 5))
    def forward(self, x): return self.layers(x)

def nas_score(config):
    torch.manual_seed(SEED); model = NASNN(config['layers'], config['width'], config['activation']).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    generator = torch.Generator(device=DEVICE).manual_seed(SEED); model.train()
    for _ in range(SEARCH_EPOCHS):
        order = torch.randperm(len(x_train_tensor), generator=generator, device=DEVICE)
        for start in range(0, len(order), config['batch_size']):
            index = order[start:start + config['batch_size']]; optimizer.zero_grad()
            loss = nn.CrossEntropyLoss()(model(x_train_tensor[index]), y_train_tensor[index]); loss.backward(); optimizer.step()
    with torch.no_grad(): prediction = model(x_validation_tensor).argmax(dim=1).cpu().numpy()
    return f1_score(y_validation, prediction, average='macro'), balanced_accuracy_score(y_validation, prediction), sum(p.numel() for p in model.parameters())

rng = np.random.default_rng(SEED); population = [{name: values[rng.integers(len(values))] for name, values in NAS_CHOICES.items()} for _ in range(4)]; nas_rows = []
for generation in range(1, 4):
    scored = []
    for config in population:
        started = time.perf_counter(); macro_f1, balanced_accuracy, parameters = nas_score(config)
        row = {**config, 'trial': len(nas_rows) + 1, 'generation': generation, 'macro_f1': macro_f1, 'balanced_accuracy': balanced_accuracy, 'parameters': parameters, 'seconds': time.perf_counter() - started}; nas_rows.append(row); scored.append(row)
    scored.sort(key=lambda row: row['macro_f1'], reverse=True)
    if generation < 3:
        population = [{name: scored[0][name] for name in NAS_CHOICES}]
        while len(population) < 4:
            parent_a, parent_b = (scored[index] for index in rng.choice(len(scored), size=2, replace=False))
            child = {name: parent_a[name] if rng.random() < 0.5 else parent_b[name] for name in NAS_CHOICES}
            for name, values in NAS_CHOICES.items():
                if rng.random() < 0.2: child[name] = values[rng.integers(len(values))]
            population.append(child)
nas_table = pd.DataFrame(nas_rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
display(nas_table)


### 4.7 Best configuration, performance, and search time

| Item | Value |
|---|---:|
| Hidden layers | 3 |
| Hidden width | 48 |
| Activation | Sigmoid |
| Learning rate | 0.01 |
| Batch size | 512 |
| Weight decay | 0.0 |
| Epochs | 20 |
| Validation macro-F1 | 0.7404376107814185 |
| Validation balanced accuracy | 0.6900639269431423 |
| Trainable parameters | 5669 |
| Search time on Tesla T4 x2 (seconds) | 259.625933563 |


### 4.8 Search results table

| Rank | Trial | Layers | Width | Activation | Learning rate | Batch size | Weight decay | Macro-F1 | Balanced accuracy | Parameters | Seconds |
|---:|---:|---:|---:|---|---:|---:|---:|---:|---:|---:|---:|
| 1 | 9 | 3 | 48 | Sigmoid | 0.01 | 512 | 0.0 | 0.7404376107814185 | 0.6900639269431423 | 5669 | 20.59312578800001 |
| 2 | 12 | 3 | 48 | Sigmoid | 0.01 | 512 | 0.0 | 0.7404376107814185 | 0.6900639269431423 | 5669 | 20.08412137099998 |
| 3 | 7 | 3 | 48 | Sigmoid | 0.01 | 512 | 0.0 | 0.7404376107814185 | 0.6900639269431423 | 5669 | 20.827572151000027 |
| 4 | 3 | 3 | 48 | ReLU | 0.01 | 512 | 0.0 | 0.7296520063979381 | 0.6735984669272197 | 5669 | 20.702719058000014 |
| 5 | 5 | 3 | 48 | ReLU | 0.01 | 512 | 0.0 | 0.7296520063979381 | 0.6735984669272197 | 5669 | 20.14614876899998 |
| 6 | 8 | 3 | 64 | ReLU | 0.001 | 512 | 0.001 | 0.6325100061565934 | 0.5764284760871896 | 9605 | 21.08265175400004 |
| 7 | 10 | 3 | 64 | ReLU | 0.01 | 512 | 0.001 | 0.5965193926675724 | 0.5408782309861777 | 9605 | 21.262919695999983 |
| 8 | 4 | 3 | 32 | ReLU | 0.001 | 256 | 0.001 | 0.5765014429582862 | 0.5299910709057312 | 2757 | 42.257107924000024 |
| 9 | 6 | 1 | 48 | ReLU | 0.001 | 512 | 0.001 | 0.4555485214382717 | 0.4322894170036835 | 965 | 15.526241975000005 |
| 10 | 1 | 1 | 48 | ReLU | 0.001 | 512 | 0.001 | 0.4555485214382717 | 0.4322894170036835 | 965 | 21.166643714999992 |
| 11 | 11 | 3 | 48 | Sigmoid | 0.0001 | 512 | 0.0 | 0.35863708083432466 | 0.3480249902095238 | 5669 | 20.695660927999995 |
| 12 | 2 | 1 | 48 | Sigmoid | 0.0001 | 512 | 0.001 | 0.317453975902176 | 0.31968304537043174 | 965 | 15.242923937 |


## Task 5 — Performance comparison and analysis


### 5.1 Compare primary and secondary metrics

The comparison retrains each selected configuration on the same training split and evaluates macro-F1 and balanced accuracy on the same held-out test split.


### 5.2 Search time, trials, and model complexity

The comparison table includes trial count, search time, retraining time, and trainable parameter count.


### 5.3 Summary table and comparison visualisation

| Method | Test macro-F1 | Test balanced accuracy | Trials | Search seconds | Retrain seconds | Parameters |
|---|---:|---:|---:|---:|---:|---:|
| Baseline | 0.4506740035305746 | 0.4354210015421162 | 1 | 0.0 | 19.637850962000016 | 725 |
| Random search | 0.6174767666159122 | 0.5597810527398397 | 12 | 220.10828787800006 | 20.100843670000017 | 725 |
| Bayesian optimisation | 0.6174767666159122 | 0.5597810527398397 | 12 | 228.18862507699998 | 19.671598487999972 | 725 |
| Evolutionary NAS | 0.7392333120930268 | 0.6893981687503743 | 12 | 259.625933563 | 22.314309587000025 | 5669 |

The figure is saved as `figures/performance_comparison.png`.


In [ ]:
# Refit every validation-selected configuration once, then evaluate the untouched test split.
class NASNN(nn.Module):
    def __init__(self, layers, width, activation):
        super().__init__()
        activation_layer = nn.Sigmoid if activation == 'Sigmoid' else nn.ReLU
        network, previous = [], len(feature_names)
        for _ in range(layers):
            network += [nn.Linear(previous, width), activation_layer()]
            previous = width
        self.layers = nn.Sequential(*network, nn.Linear(previous, 5))
    def forward(self, x):
        return self.layers(x)

def fit_and_score(config, nas=False, final=False):
    frame = train_validation_frame if final else train_frame
    fitted_scaler = StandardScaler().fit(frame[continuous_features])
    def features(source):
        values = source[feature_names].astype('float32').copy()
        values[continuous_features] = fitted_scaler.transform(values[continuous_features])
        return values.to_numpy(), source[target].to_numpy(dtype=np.int64) - 1
    train_x, train_y = features(frame)
    test_x, test_y = features(test_frame)
    torch.manual_seed(SEED)
    model = NASNN(config['layers'], config['width'], config['activation']).to(DEVICE) if nas else BaselineNN(len(feature_names)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    train_x, train_y = torch.tensor(train_x, dtype=torch.float32, device=DEVICE), torch.tensor(train_y, dtype=torch.long, device=DEVICE)
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    model.train()
    for _ in range(config['epochs']):
        order = torch.randperm(len(train_x), generator=generator, device=DEVICE)
        for start in range(0, len(train_x), config['batch_size']):
            index = order[start:start + config['batch_size']]
            optimizer.zero_grad(); loss = nn.CrossEntropyLoss()(model(train_x[index]), train_y[index]); loss.backward(); optimizer.step()
    with torch.no_grad():
        prediction = model(torch.tensor(test_x, dtype=torch.float32, device=DEVICE)).argmax(dim=1).cpu().numpy()
    return model, fitted_scaler, test_y, prediction

selected = [
    ('Baseline', {'learning_rate': 0.001, 'batch_size': 512, 'weight_decay': 0.0, 'epochs': 20}, False, 1, 0.0),
    ('Random search', {'learning_rate': 0.01, 'batch_size': 512, 'weight_decay': 0.00001, 'epochs': 20}, False, 12, 220.10828787800006),
    ('Bayesian optimisation', {'learning_rate': 0.01, 'batch_size': 512, 'weight_decay': 0.00001, 'epochs': 20}, False, 12, 228.18862507699998),
    ('Evolutionary NAS', {'layers': 3, 'width': 48, 'activation': 'Sigmoid', 'learning_rate': 0.01, 'batch_size': 512, 'weight_decay': 0.0, 'epochs': 20}, True, 12, 259.625933563),
]
comparison_rows = []
for name, config, nas, trials, search_seconds in selected:
    started = time.perf_counter(); model, _, labels, prediction = fit_and_score(config, nas)
    comparison_rows.append({'Method': name, 'Test macro-F1': f1_score(labels, prediction, average='macro'), 'Test balanced accuracy': balanced_accuracy_score(labels, prediction), 'Trials': trials, 'Search seconds': search_seconds, 'Retrain seconds': time.perf_counter() - started, 'Parameters': sum(parameter.numel() for parameter in model.parameters())})
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)
print(comparison_table.to_csv(index=False))
comparison_table.to_csv(ROOT / 'task5_comparison.csv', index=False)
comparison_table.plot(x='Method', y=['Test macro-F1', 'Test balanced accuracy'], kind='bar', ylim=(0, 1), rot=15)
plt.tight_layout(); plt.savefig(FIGURE_PATH, dpi=150); plt.show()

# NAS wins validation selection, so retrain it on the combined 80% pool before one final test evaluation.
final_config = selected[-1][1]
final_config = {**final_config, 'epochs': FINAL_EPOCHS}
final_model, final_scaler, final_labels, final_prediction = fit_and_score(final_config, nas=True, final=True)
torch.save({'state_dict': final_model.state_dict(), 'architecture': {'hidden_layers': [final_config['width']] * final_config['layers'], 'activation': final_config['activation']}, 'feature_names': feature_names, 'continuous_features': continuous_features, 'scaler_mean': final_scaler.mean_, 'scaler_scale': final_scaler.scale_}, MODEL_PATH)
print(classification_report(final_labels, final_prediction, digits=16))
print('Final test macro-F1:', f1_score(final_labels, final_prediction, average='macro'))
print('Final test balanced accuracy:', balanced_accuracy_score(final_labels, final_prediction))
print('Saved model:', MODEL_PATH)


### 5.4 Hyperparameter or architecture effects

The best NAS candidate used three 48-unit Sigmoid layers with learning rate 0.01 and no weight decay. It increased validation macro-F1 from 0.620486164019339 for the fixed baseline architecture searches to 0.7404376107814185, at the cost of 5669 rather than 725 parameters.


### 5.5 Fairness of the comparison

All methods use the same seed, preprocessing rule, data splits, metric, and epoch budget. Search spaces and trial budgets differ by method, and NAS also changes architecture.


### 5.6 Final model selection and complete configuration

The final method is selected by validation macro-F1, not by held-out test performance. It is then retrained on the combined 80% training and validation pool for 100 epochs. Its configuration is: three hidden layers, width 48, Sigmoid activation, learning rate 0.01, batch size 512, and weight decay 0.0.


### 5.7 Class-level evaluation, limitations, and saved model

The selected NAS model is retrained on the combined 80% training and validation pool for 100 epochs. Its held-out test macro-F1 is 0.804469009937745 and balanced accuracy is 0.7509017561327646. Class 3 remains the main limitation: its recall is 0.501953125. The exact model, architecture, feature order, and fitted scaler are saved to `models/1173808_Assignment2_final.pt`.


## Task 6 — Hidden test


### 6.1 Run the final model on a hidden-test CSV and create Cover_Type predictions

Run `python predict.py data/input.csv data/predictions.csv` from the repository root. The script loads `models/1173808_Assignment2_final.pt`, validates the 14 feature columns, applies the saved scaler, and writes one `Cover_Type` prediction per row.
